# Analisi — Holdout 70/30 + DeLong appaiato

**Domanda:** aggiungere **sesso + età** migliora il rilevamento del PDAC rispetto al solo
dato-immagine? Qui usiamo **un singolo split** stratificato 70% train / 30% test, e per ogni
modello confrontiamo *solo-immagine* vs *immagine+clinica* con il **test di DeLong appaiato**.

> Questo notebook è **autoconsistente**: tutta la logica è qui dentro, cella per cella.
> Non importa nulla da `panorama_ml`. Kernel: **PANORAMA ML (.venv-ml)**.
> ⚠️ Su ~98 casi di test un singolo split è instabile: per il pilota è più affidabile il
> notebook 03 (CV ripetuta). Il 70/30 avrà senso pieno sul dataset completo (2238).

In [1]:
# --- Setup e RIPRODUCIBILITÀ -------------------------------------------------
# Fissiamo tutti i semi casuali: rieseguendo si ottengono SEMPRE gli stessi numeri.
import os, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)
print("Semi casuali fissati a", SEED)

# Trova la radice del progetto (funziona da qualunque cartella: notebooks/, ml_study/, radice).
# Cerca la cartella che contiene le feature e/o il file clinico -> così l'analisi gira
# anche copiando solo ml_study/ + cache/clinical_information.xlsx (senza imagesTr).
CWD = Path.cwd()
def _e_radice(p):
    return (p / "ml_study" / "cache_features").exists() or (p / "cache" / "clinical_information.xlsx").exists()
PROJECT_DIR = next((p for p in [CWD, *CWD.parents] if _e_radice(p)), CWD)
CLINICAL   = PROJECT_DIR / "cache" / "clinical_information.xlsx"
CACHE_FEAT = PROJECT_DIR / "ml_study" / "cache_features"
print("Radice progetto:", PROJECT_DIR)
print("File clinico presente:", CLINICAL.exists(), "| cartella feature:", CACHE_FEAT.exists())

Semi casuali fissati a 42
Radice progetto: D:\PANORAMA_v2
File clinico presente: True | cartella feature: True


## 1) La coorte: 326 casi
Costruiamo l'elenco dei pazienti dello studio. Regola: teniamo solo i casi **scaricati**
che hanno **sesso ed età** (le due variabili cliniche che vogliamo testare). Questo esclude
automaticamente i sottoinsiemi pubblici NIH e MSD (privi di quei metadati) → restano i due
centri olandesi RUMC + UMCG.

- **`y`** = 1 se PDAC, 0 se controllo
- **`sex_bin`** = 0 femmina, 1 maschio
- **`age`** = età in anni (il file clinico la salva come `'077Y'`, la convertiamo in 77)

In [2]:
# --- Costruzione coorte (INLINE) --------------------------------------------
df = pd.read_excel(CLINICAL).rename(columns={"PANORAMA_study_id": "study_id"})

# età: '077Y' -> 77.0
df["age"] = pd.to_numeric(df["patient_age"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
# sesso -> 0/1
sx = df["patient_sex"].astype(str).str.strip().str.upper().str[0]
df["sex_bin"] = sx.map({"F": 0, "M": 1})
# etichetta PDAC
df["y"] = (df["label"].astype(str).str.upper() == "PDAC").astype(int)

# solo casi con feature già estratte (usiamo il file radiomica come riferimento dei presenti)
present = set(pd.read_parquet(CACHE_FEAT / "feat_radiomics.parquet")["study_id"])
cohort = df[df["study_id"].isin(present) & df["sex_bin"].notna() & df["age"].notna()].copy()
cohort = cohort.drop_duplicates("study_id")[["study_id", "y", "age", "sex_bin", "label", "level"]].reset_index(drop=True)
assert cohort["study_id"].is_unique, "study_id duplicati nella coorte!"

print("Casi nella coorte:", len(cohort))
print("PDAC vs controlli:\n", cohort["y"].map({1: "PDAC", 0: "controllo"}).value_counts().to_string())
print("\nSesso per gruppo:")
print(pd.crosstab(cohort["y"].map({1: "PDAC", 0: "controllo"}), cohort["sex_bin"].map({0: "F", 1: "M"})).to_string())
for g, s in cohort.groupby(cohort["y"].map({1: "PDAC", 0: "controllo"})):
    print(f"  età {g}: mediana {s['age'].median():.0f} (range {s['age'].min():.0f}-{s['age'].max():.0f})")
cohort.head()

Casi nella coorte: 326
PDAC vs controlli:
 y
PDAC         182
controllo    144

Sesso per gruppo:
sex_bin     F   M
y                
PDAC       91  91
controllo  73  71
  età PDAC: mediana 71 (range 41-89)
  età controllo: mediana 64 (range 23-99)


,study_id,y,age,sex_bin,label,level
0,100012_00001,0,73.0,1.0,non-PDAC,radiology
1,100030_00001,1,51.0,0.0,PDAC,histopathology
2,100031_00001,0,38.0,1.0,non-PDAC,radiology
3,100037_00001,0,23.0,1.0,non-PDAC,radiology
4,100043_00001,1,73.0,0.0,PDAC,pathology


## 2) Le feature-immagine (già estratte, in cache)
Per ogni paziente abbiamo un vettore di numeri che riassume la sua TC, prodotto da diversi
**estrattori**. Sono già stati calcolati e salvati in `cache_features/*.parquet` (vedi il
notebook `01_come_si_estraggono_le_features`). Qui li **carichiamo e basta** — sono dati puri.

| Estrattore | Cos'è |
|---|---|
| `radiomics` | feature pyradiomics (forma + intensità + texture) dalla ROI |
| `merlin` | embedding di un foundation model di **CT addominale** (Stanford) |
| `ctfm`, `medicalnet`, `models_genesis`, `stu_net` | embedding di foundation model 3D |
| `imagenet`, `radimagenet` | embedding di backbone 2D (foto naturali / radiologia) |

In [3]:
# --- Carica le feature dai parquet (INLINE) ---------------------------------
NOMI = ["radiomics", "merlin", "merlin_roi", "ctfm", "medicalnet", "models_genesis",
        "stu_net", "imagenet", "radimagenet"]
features = {}
for nome in NOMI:
    p = CACHE_FEAT / f"feat_{nome}.parquet"
    if p.exists():
        features[nome] = pd.read_parquet(p)
        print(f"  {nome:16s}: {features[nome].shape[1]-1:4d} feature, {len(features[nome])} casi")
    else:
        print(f"  {nome:16s}: MANCANTE (verrà saltato)")
print("\nEstrattori disponibili:", list(features))

  radiomics       :  107 feature, 326 casi
  merlin          : 2048 feature, 326 casi
  merlin_roi      : 2048 feature, 326 casi
  ctfm            :  512 feature, 326 casi


  medicalnet      : 2048 feature, 326 casi
  models_genesis  :  512 feature, 326 casi
  stu_net         :  512 feature, 326 casi
  imagenet        : 2048 feature, 326 casi
  radimagenet     : 2048 feature, 326 casi

Estrattori disponibili: ['radiomics', 'merlin', 'merlin_roi', 'ctfm', 'medicalnet', 'models_genesis', 'stu_net', 'imagenet', 'radimagenet']


## 3) Gli strumenti (definiti qui, così li vedi)
Definiamo — **inline** — le funzioni che useremo:
- **`build_Xy`**: allinea le feature alla coorte e prepara le matrici (immagine, clinica, y).
- **`make_model`**: il classificatore (regressione logistica L2, oppure random forest).
- **`run_experiment`**: dato uno split train/test, addestra e valuta TRE modelli —
  *solo-immagine*, *immagine+sesso+età*, *solo-clinica* (pavimento) — e restituisce le
  probabilità predette sul test.
- **`make_strata`**: chiave di stratificazione (malattia × sesso × fascia d'età) per split bilanciati.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

def build_Xy(feat_df, cohort):
    """Allinea le feature ALL'ORDINE della coorte (per study_id): stesse righe/ordine per
       TUTTI gli estrattori -> lo split e' identico ovunque. Ritorna X_img, clinica, y."""
    f = feat_df.drop_duplicates("study_id").set_index("study_id").reindex(cohort["study_id"])
    img_cols = [c for c in f.columns if c != "roi_used_mask"]
    X_img = f[img_cols].to_numpy(float)
    n_nan = int(np.isnan(X_img).all(axis=1).sum())
    if n_nan:
        print(f"  ⚠️ {n_nan} casi con feature tutte-NaN (estrazione fallita): verranno imputati (vedi Limiti).")
    clin = cohort[["age", "sex_bin"]].to_numpy(float)
    y = cohort["y"].to_numpy(int)
    return X_img, clin, y

def make_model(kind):
    """Impute (NaN->mediana) + standardizza + classificatore. Deterministico.
       NB: C=0.1 FISSO (non tunato) -> vedi 'Limiti'."""
    if kind == "logreg":
        clf = LogisticRegression(penalty="l2", C=0.1, max_iter=5000, class_weight="balanced")
    else:  # "rf"
        clf = RandomForestClassifier(n_estimators=400, max_features="sqrt", min_samples_leaf=3,
                                     class_weight="balanced", random_state=SEED, n_jobs=-1)
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("clf", clf)])

def run_experiment(X_img, clin, y, tr, te, kind="logreg"):
    """Addestra su tr, valuta su te i 3 modelli (senza / con clinica / solo-clinica).
       Le feature-immagine sono IDENTICHE nei bracci 'senza' e 'con' -> isola l'effetto clinico."""
    m0 = make_model(kind).fit(X_img[tr], y[tr]);     p0 = m0.predict_proba(X_img[te])[:, 1]
    Xc_tr = np.hstack([X_img[tr], clin[tr]]); Xc_te = np.hstack([X_img[te], clin[te]])
    m1 = make_model(kind).fit(Xc_tr, y[tr]);         p1 = m1.predict_proba(Xc_te)[:, 1]
    m2 = make_model("logreg").fit(clin[tr], y[tr]);  p2 = m2.predict_proba(clin[te])[:, 1]
    return dict(y_test=y[te], p_senza=p0, p_con=p1, p_clinica=p2)

def make_strata(cohort):
    """malattia x sesso x fascia d'eta' (terzili), calcolata UNA volta sulla coorte comune."""
    ab = pd.qcut(cohort["age"], 3, labels=False, duplicates="drop").astype("Int64").astype(str)
    return (cohort["y"].astype(str) + "_" + cohort["sex_bin"].astype("Int64").astype(str) + "_" + ab).to_numpy()

def auc(y_true, p):
    try: return roc_auc_score(y_true, p)
    except Exception: return np.nan

print("Funzioni pronte: build_Xy, make_model, run_experiment, make_strata, auc")

Funzioni pronte: build_Xy, make_model, run_experiment, make_strata, auc


In [5]:
# Quali modelli valutiamo: radiomica con DUE classificatori (logreg + RF), i deep con logreg.
def righe_modelli():
    for nome in features:
        if nome == "radiomics":
            yield (f"{nome} · logreg", nome, "logreg")
            yield (f"{nome} · random_forest", nome, "rf")
        else:
            yield (f"{nome} · logreg (linear probe)", nome, "logreg")

## 4) Il test statistico: DeLong appaiato

In [6]:
from scipy import stats as sps
def delong_test(y_true, s1, s2):
    """Test di DeLong APPAIATO: confronta due AUROC (con vs senza clinica) sullo STESSO
       test set. Corretto quando le due predizioni vengono dagli stessi pazienti.
       Ritorna (auc1, auc2, differenza, p-value)."""
    def midrank(x):
        J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]: j += 1
            T[i:j] = 0.5 * (i + j - 1) + 1; i = j
        T2 = np.empty(N); T2[J] = T; return T2
    y = np.asarray(y_true).astype(int); order = np.argsort(-y); ys = y[order]; m = int(ys.sum())
    P = np.vstack([np.asarray(s, float)[order] for s in (s1, s2)]); n = P.shape[1] - m
    tx = np.array([midrank(P[r, :m]) for r in range(2)])
    ty = np.array([midrank(P[r, m:]) for r in range(2)])
    tz = np.array([midrank(P[r])     for r in range(2)])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1) / 2.0 / n
    cov = np.cov((tz[:, :m] - tx) / n) / m + np.cov(1 - (tz[:, m:] - ty) / m) / n
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]; diff = aucs[0] - aucs[1]
    if var > 0:
        z = diff / np.sqrt(var); p = 2 * sps.norm.sf(abs(z))
    else:                       # varianza degenere -> nessuna evidenza (NON significativo)
        p = 1.0
    return float(aucs[0]), float(aucs[1]), float(diff), float(p)
print("Funzione delong_test pronta.")

Funzione delong_test pronta.


## 5) Eseguiamo e leggiamo i risultati
Per ogni modello: un solo split 70/30, poi AUROC *solo-immagine* vs *immagine+clinica*
sullo stesso test set, e il **p di DeLong**. In più il *pavimento* (solo sesso+età).

In [7]:
from sklearn.model_selection import StratifiedShuffleSplit
# UNO split comune, calcolato una volta sulla coorte -> identico per TUTTI gli estrattori
strata = make_strata(cohort)
tr, te = next(StratifiedShuffleSplit(1, test_size=0.30, random_state=SEED).split(np.arange(len(cohort)), strata))
righe = []
for label, nome, kind in righe_modelli():
    Xi, cl, y = build_Xy(features[nome], cohort)
    r = run_experiment(Xi, cl, y, tr, te, kind)
    a0, a1, diff, p = delong_test(r["y_test"], r["p_con"], r["p_senza"])
    righe.append(dict(modello=label,
                      AUROC_solo_img=round(auc(r["y_test"], r["p_senza"]), 3),
                      AUROC_img_clinica=round(auc(r["y_test"], r["p_con"]), 3),
                      delta=round(diff, 3), p_DeLong=round(p, 3),
                      AUROC_solo_clinica=round(auc(r["y_test"], r["p_clinica"]), 3),
                      n_test=len(te)))
tab = pd.DataFrame(righe).sort_values("AUROC_solo_img", ascending=False).reset_index(drop=True)
tab

,modello,AUROC_solo_img,AUROC_img_clinica,delta,p_DeLong,AUROC_solo_clinica,n_test
0,merlin · logreg (linear probe),0.872,0.873,0.001,0.211,0.656,98
1,radiomics · logreg,0.854,0.874,0.020,0.017,0.656,98
2,radiomics · random_forest,0.833,0.846,0.013,0.153,0.656,98
3,stu_net · logreg (linear probe),0.823,0.822,-0.001,0.717,0.656,98
4,merlin_roi · logreg (linear probe),0.811,0.813,0.002,0.632,0.656,98
5,ctfm · logreg (linear probe),0.752,0.762,0.011,0.232,0.656,98
6,medicalnet · logreg (linear probe),0.746,0.747,0.001,0.771,0.656,98
7,radimagenet · logreg (linear probe),0.734,0.736,0.003,0.486,0.656,98
8,imagenet · logreg (linear probe),0.729,0.735,0.006,0.069,0.656,98
9,models_genesis · logreg (linear probe),0.703,0.709,0.006,0.283,0.656,98


In [8]:
sig = tab[tab["p_DeLong"] < 0.05]
print("Miglioramenti SIGNIFICATIVI (p<0.05):", list(sig["modello"]) if len(sig) else "nessuno (atteso sul pilota)")
print("Δ medio della clinica sugli estrattori:", round(tab["delta"].mean(), 3))

Miglioramenti SIGNIFICATIVI (p<0.05): ['radiomics · logreg']
Δ medio della clinica sugli estrattori: 0.006


## 6) Come interpretare (onestà)
- Confronta **`AUROC_solo_img`** con **`AUROC_img_clinica`**: se la seconda è più alta la clinica
  *sembra* aiutare; **`p`** < 0.05 → differenza statisticamente significativa.
- **`AUROC_solo_clinica`** = il *pavimento* (quanto segnale c'è in sesso+età da soli).
- **Aspettativa onesta sul pilota (326 casi):** con un effetto piccolo e pochi casi, è normale
  ottenere risultati **non significativi**. Guarda la **coerenza della direzione** tra estrattori:
  è il segnale più affidabile. La risposta statistica vera arriverà sul **dataset completo (2238)**.

## ⚠️ Limiti (da leggere)
Questo è un **pilota su 326 casi**; i risultati vanno letti con questi limiti (emersi da una revisione del codice):

1. **Confronto tra estrattori non "apples-to-apples"**: i modelli usano input diversi (Merlin sul volume intero; gli altri sul box pancreas; ImageNet/RadImageNet in 2.5D) e feature di dimensione diversa. Per questo abbiamo aggiunto **`merlin_roi`** (Merlin anche sul box). La classifica *tra* estrattori è indicativa; il confronto valido è **dentro la riga** (con vs senza clinica).
2. **Regolarizzazione fissa `C=0.1` (non tunata)** applicata a modelli con 2 / 107 / 2048 feature → rappresentazioni confrontate a un punto operativo arbitrario e dipendente dalla dimensione.
3. **"La clinica non aggiunge nulla" è testato solo con early-fusion** (concatenazione): con 2 variabili cliniche fra centinaia/migliaia di feature-immagine sotto L2, l'effetto clinico può essere "annegato". Un test più pulito (non incluso) sarebbe la **late-fusion** (punteggio-immagine + clinica). L'incremento è inoltre **a bassa potenza**.
4. **Confronti multipli non corretti**: ~9 righe testate a α=0.05 senza FDR/Holm → attesi ~0.5 falsi positivi. Un singolo "significativo" isolato è **verosimilmente casuale**.
5. **Holdout 70/30 (notebook 3) instabile** su ~98 casi di test → per il pilota fidati della **CV ripetuta (notebook 4)**.
6. La risposta **definitiva** richiede il **dataset completo (2238 casi)**.